In [7]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import quote

import numpy as np

os.environ["POLARS_OOC_MEMORY_BUDGET_MB"] = "100"  # disable Polars' own memory budget
os.environ["POLARS_ENGINE_AFFINITY"] = "streaming"
os.environ["POLARS_STREAMING_CHUNK_SIZE"] = "2000"  # small chunks to stress memory
os.environ["POLARS_VERBOSE"] = "0"  # enable verbose logging to stdout for debugging
os.environ["POLARS_MAX_THREADS"] = "1"  # enable verbose logging to stdout for debugging

import polars as pl

_IMPL = Path(".").parent / "_memory_analysis_impl.py"
assert _IMPL.exists(), f"Expected {_IMPL} to exist"


def write_partitioned_measurements(src: Path, n_partitions: int, rows_per_partition: int) -> float:
    """Write partitioned parquet with (measurement, unit, timestamp, value) columns.

    Each partition holds ``ROWS_PER_PARTITION`` readings spread over
    ``N_MEASUREMENTS`` distinct measurements, so grouping by measurement yields
    a handful of groups with very long ``timestamp``/``value`` list columns.
    """
    total_mbytes = 0
    for p in range(n_partitions):
        measurement = f"measurement_{p}"
        measurement_col = "measurement"
        partition_dir = src / f"{quote(measurement_col)}={quote(measurement)}"
        partition_dir.mkdir(parents=True, exist_ok=True)
        df = pl.DataFrame([
            pl.repeat(measurement, rows_per_partition, eager=True).rename(measurement_col),
            (pl.int_range(0, rows_per_partition, eager=True) % 5)
            .cast(pl.String)
            .str.pad_start(length=3, fill_char="0")
            .rename("channel"),
            pl.Series(np.random.rand(rows_per_partition)).rename("value"),
        ])
        df.write_parquet(partition_dir / "00001.parquet")
        total_mbytes += df.estimated_size(unit="mb")
    return total_mbytes


def _run(name: str, tmp_path: Path, n_partitions: int, rows_per_partition: int) -> dict:
    """Spawn a fresh interpreter to run ``run_<name>`` in ``_memory_analysis_impl.py``."""
    src = tmp_path / "src"
    src.mkdir(parents=True, exist_ok=True)
    data_size_mb = write_partitioned_measurements(src, n_partitions, rows_per_partition)

    result = subprocess.run(
        [
            sys.executable,
            str(_IMPL),
            name,
            str(tmp_path),
            str(rows_per_partition),
        ],
        capture_output=True,
        text=True,
        env={**os.environ},
    )
    output = (result.stdout + result.stderr).strip()
    print(f"Output from memory worker '{name}':\n{output}")

    if result.returncode != 0:
        raise RuntimeError(f"Memory worker '{name}' failed:\n{output}")
    else:
        data = json.loads(output.splitlines()[-1])
        data.update({"dataset_size_mb": data_size_mb})
        return data


rows_per_partition = 100_000
num_partitions = 200

In [8]:
results_pure_polars = []

for i in range(1, num_partitions + 1, 20):
    with TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        result = _run("pure_polars", tmp_path, i, rows_per_partition)
        print(
            f"Peak RSS: {result['peak_rss_mb']:.1f} MB, Dataset size: {result['dataset_size_mb']:.1f} MB"
        )

        result.update({"n_partitions": i, "rows_per_partition": rows_per_partition})
        results_pure_polars.append(result)

results_polars_df = pl.DataFrame(results_pure_polars)

Output from memory worker 'pure_polars':
{"peak_rss_mb": 40.75}
Peak RSS: 40.8 MB, Dataset size: 2.3 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 199.0625}
Peak RSS: 199.1 MB, Dataset size: 49.1 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 273.96875}
Peak RSS: 274.0 MB, Dataset size: 96.8 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 331.40625}
Peak RSS: 331.4 MB, Dataset size: 144.5 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 391.84375}
Peak RSS: 391.8 MB, Dataset size: 192.2 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 473.375}
Peak RSS: 473.4 MB, Dataset size: 239.9 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 512.09375}
Peak RSS: 512.1 MB, Dataset size: 289.5 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 592.53125}
Peak RSS: 592.5 MB, Dataset size: 339.1 MB
Output from memory worker 'pure_polars':
{"peak_rss_mb": 623.828125}
Peak RSS: 623.8 MB, Dataset size: 388.7 MB
Output from 

In [9]:
results_all = []

for i in range(1, num_partitions + 1, 20):
    with TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        result = _run("all", tmp_path, i, rows_per_partition)
        print(
            f"Peak RSS: {result['peak_rss_mb']:.1f} MB, Dataset size: {result['dataset_size_mb']:.1f} MB"
        )

        result.update({"n_partitions": i, "rows_per_partition": rows_per_partition})
        results_all.append(result)

results_all_df = pl.DataFrame(results_all)

Output from memory worker 'all':
{"peak_rss_mb": 109.21875}
Peak RSS: 109.2 MB, Dataset size: 2.3 MB
Output from memory worker 'all':
{"peak_rss_mb": 750.953125}
Peak RSS: 751.0 MB, Dataset size: 49.1 MB
Output from memory worker 'all':
{"peak_rss_mb": 1419.9375}
Peak RSS: 1419.9 MB, Dataset size: 96.8 MB
Output from memory worker 'all':
{"peak_rss_mb": 1882.84375}
Peak RSS: 1882.8 MB, Dataset size: 144.5 MB
Output from memory worker 'all':
{"peak_rss_mb": 2156.15625}
Peak RSS: 2156.2 MB, Dataset size: 192.2 MB
Output from memory worker 'all':
{"peak_rss_mb": 2609.546875}
Peak RSS: 2609.5 MB, Dataset size: 239.9 MB
Output from memory worker 'all':
{"peak_rss_mb": 2688.78125}
Peak RSS: 2688.8 MB, Dataset size: 289.5 MB
Output from memory worker 'all':
{"peak_rss_mb": 3121.0}
Peak RSS: 3121.0 MB, Dataset size: 339.1 MB
Output from memory worker 'all':
{"peak_rss_mb": 3519.0625}
Peak RSS: 3519.1 MB, Dataset size: 388.7 MB
Output from memory worker 'all':
{"peak_rss_mb": 3837.203125}
Peak 

In [10]:
results_by_partition = []

for i in range(1, num_partitions + 1, 20):
    with TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        result = _run("by_partition", tmp_path, i, rows_per_partition)
        print(
            f"Peak RSS: {result['peak_rss_mb']:.1f} MB, Dataset size: {result['dataset_size_mb']:.1f} MB"
        )

        result.update({"n_partitions": i, "rows_per_partition": rows_per_partition})
        results_by_partition.append(result)

results_by_partition_df = pl.DataFrame(results_by_partition)

Output from memory worker 'by_partition':
{"peak_rss_mb": 118.765625}
Peak RSS: 118.8 MB, Dataset size: 2.3 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 211.203125}
Peak RSS: 211.2 MB, Dataset size: 49.1 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 269.53125}
Peak RSS: 269.5 MB, Dataset size: 96.8 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 271.609375}
Peak RSS: 271.6 MB, Dataset size: 144.5 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 308.296875}
Peak RSS: 308.3 MB, Dataset size: 192.2 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 310.625}
Peak RSS: 310.6 MB, Dataset size: 239.9 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 335.875}
Peak RSS: 335.9 MB, Dataset size: 289.5 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 321.109375}
Peak RSS: 321.1 MB, Dataset size: 339.1 MB
Output from memory worker 'by_partition':
{"peak_rss_mb": 324.859375}
Peak RSS: 324.9 MB, Dataset size: 388

In [18]:
from polars import selectors as cs

results_df = (
    results_polars_df
    .select(
        "n_partitions",
        "rows_per_partition",
        "dataset_size_mb",
        pl.exclude("n_partitions", "rows_per_partition", "dataset_size_mb").name.suffix("_polars"),
    )
    .join(
        results_all_df,
        on=("n_partitions", "rows_per_partition", "dataset_size_mb"),
        how="full",
        suffix="_all",
        coalesce=True,
    )
    .join(
        results_by_partition_df,
        on=("n_partitions", "rows_per_partition", "dataset_size_mb"),
        how="full",
        suffix="_by_partition",
        coalesce=True,
    )
)

results_df.unpivot(
    index="n_partitions",
    on=cs.starts_with("peak_rss_mb") | cs.starts_with("dataset_size_mb"),
    variable_name="metric",
    value_name="value",
).plot.line().encode(x="n_partitions", y="value", color="metric")

alt.Chart(...)

In [6]:
results_df

n_partitions,peak_rss_mb_polars,dataset_size_mb_polars,rows_per_partition_polars,peak_rss_mb,dataset_size_mb,rows_per_partition,peak_rss_mb_by_partition,dataset_size_mb_by_partition,rows_per_partition_by_partition
i64,f64,f64,i64,f64,f64,i64,f64,f64,i64
1,0.0,0.228882,10000000,58.640625,0.228882,10000000,61.21875,0.228882,10000000
21,78.515625,4.911423,10000000,184.890625,4.911423,10000000,127.09375,4.911423,10000000
41,104.625,9.679794,10000000,279.5,9.679794,10000000,164.671875,9.679794,10000000
61,125.015625,14.448166,10000000,351.96875,14.448166,10000000,186.4375,14.448166,10000000
81,143.046875,19.216537,10000000,423.40625,19.216537,10000000,202.125,19.216537,10000000
101,154.40625,23.994446,10000000,492.1875,23.994446,10000000,232.671875,23.994446,10000000
121,171.125,28.953552,10000000,571.390625,28.953552,10000000,220.203125,28.953552,10000000
141,181.59375,33.912659,10000000,648.03125,33.912659,10000000,217.625,33.912659,10000000
161,197.546875,38.871765,10000000,700.828125,38.871765,10000000,231.53125,38.871765,10000000
